In [38]:
#task 4.1
import sqlite3

connection = sqlite3.connect('holiday_company.db')


connection.execute("DROP TABLE IF EXISTS Booking")
connection.execute("DROP TABLE IF EXISTS Villa")
connection.commit()


connection.execute("""
CREATE TABLE Villa (
	villaID	TEXT,
	villaName	TEXT,
	country	TEXT,
	cost	REAL,
	PRIMARY KEY(villaID)
);""")

connection.execute("""
CREATE TABLE Booking (
	bookingID	TEXT,
	customerID	TEXT,
	villaID	TEXT,
	start_date	TEXT,
	number_of_days	INTEGER,
	PRIMARY KEY(bookingID)
);""")


connection.commit()
connection.close()

In [39]:
#task 4.2
connection =  sqlite3.connect('holiday_company.db')

#
openfile = open('villas.txt', 'r')
data = openfile.readlines()
openfile.close()

for i in data:
    ID,name, country, cost = i.split(',')
    cost = cost.strip()
    
    connection.execute('INSERT INTO Villa(villaID, villaName, country, cost) VALUES (?, ? ,? ,?)', (ID, name, country, float(cost)))
    
#
openfile = open('customerBookings.txt', 'r')
data = openfile.readlines()
openfile.close()

for i in data:
    ID,customerID, villaID, start_date, number = i.split(',')
    number = number.strip()
    
    connection.execute('INSERT INTO Booking(bookingID, customerID, villaID, start_date, number_of_days) VALUES (?, ? ,? ,?,?)', (ID,customerID, villaID, start_date, int(number)))
    

    
connection.commit()

print(connection.execute('SELECT * FROM Villa').fetchall())
print(connection.execute('SELECT * FROM Booking').fetchall())
connection.close()




[('1', 'Rose', 'France', 128.0), ('2', 'Sea view', 'Australia', 325.0), ('3', 'Dolphin', 'New Zealand', 490.0), ('4', 'Flower haven', 'Mexico', 580.0), ('5', 'Mountain breeze', 'India', 268.0), ('6', 'Sunset', 'UK', 136.0), ('7', 'Moonlight', 'USA', 358.0), ('8', 'White brick', 'Italy', 410.0), ('9', 'Blue house', 'Germany', 400.0), ('10', 'Walled garden', 'Croatia', 258.0)]
[('1', '857', '3', '05-Jan', 7), ('2', '1149', '10', '20-Jan', 11), ('3', '388', '9', '05-Feb', 2), ('4', '230', '4', '03-Mar', 14), ('5', '1254', '6', '19-Nov', 10), ('6', '1500', '4', '12-May', 4), ('7', '1687', '7', '03-Mar', 6), ('8', '1408', '2', '18-Aug', 2), ('9', '1138', '3', '02-Mar', 4), ('10', '235', '5', '04-Apr', 7), ('11', '4', '10', '07-Jul', 12), ('12', '1981', '8', '15-Jan', 5), ('13', '500', '9', '06-Nov', 4), ('14', '332', '9', '03-Apr', 7), ('15', '1512', '5', '12-Oct', 14), ('16', '660', '4', '08-Jul', 6), ('17', '738', '4', '09-Aug', 7), ('18', '1211', '1', '12-Nov', 3), ('19', '1089', '1', '1

In [40]:
import sqlite3

connection = sqlite3.connect('holiday_company.db')
connection.execute("""DROP TABLE IF EXISTS Villa_Booking""")

connection.execute("""
CREATE TABLE Villa_Booking (
	villaID	TEXT,
	date	TEXT
);""")

connection.commit()

villa_dates = connection.execute("""SELECT Booking.villaID, Booking.number_of_days, Booking.start_date
FROM Booking
 """).fetchall()

Months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


def checkmaxday(check):
    if check % 2 == 1:
        maxday = 31
        
        
    elif check == 2:
        maxday = 28
    else:
        maxday =30
        
    return maxday




for i in villa_dates:
    villaid, number_of_days, startdate = i[0], i[1], i[2]
    #print(villaid)
   #print(number_of_days)
    #print(startdate)
    
    
    
    
    
    startday, startmonth = startdate.split('-')
    check = Months.index(startmonth) + 1
    
    maxday = checkmaxday(check)

    
    
    temp = startday
    curday = int(temp) 
    for i in range(number_of_days):
        
        
        if curday > maxday: #32/31
            curday = curday-maxday
            startmonth = Months[(Months.index(startmonth) + 1) % 12]
            check = Months.index(startmonth) + 1
            maxday = checkmaxday(check)
            temp = 1
            
        date = str(curday) + '-' + startmonth
        
        curday = curday + 1
        
        connection.execute('INSERT INTO Villa_Booking(villaID, date) VALUES (?,?)',(villaid, date) )
        connection.commit()
        
        
    
        
        
    
    
    
    
 






connection.close()

In [44]:
#

def display(villaname, startdate, numberofdays):
    connection = sqlite3.connect('holiday_company.db')
    
    startday, startmonth = startdate.split('-')
    check = Months.index(startmonth) + 1
    
    maxday = checkmaxday(check)

    possibledate = []
    notpossibledate = []
    
    
    temp = startday
    curday = int(temp) 
    for i in range(numberofdays):
        
        
        if curday > maxday: #32/31
            curday = curday-maxday
            startmonth = Months[(Months.index(startmonth) + 1) % 12]
            check = Months.index(startmonth) + 1
            maxday = checkmaxday(check)
            temp = 1
            
        date = str(curday) + '-' + startmonth
        
        curday = curday + 1
        
        check = connection.execute("""
        
        SELECT *
        FROM Villa_Booking, Villa
        WHERE Villa_Booking.date = ? and Villa_Booking.villaID = Villa.villaID and Villa.villaName = ? """, (date, villaname)).fetchall()
        
        
        
        if len(check) == 0:
            possibledate.append(date)
            
        else:
            notpossibledate.append(date)
            
        
            
    connection.close()
    return possibledate, notpossibledate


name = input('Villa name              ')
Month = input('Month                  ')
Date = input('Date                    ')
Numberofdays = input('Number of days: ')

date = Date + '-' + Month


possible, impossible = display(name, date, int(Numberofdays))


print('Dates that are available')
for i in possible:
    print(i)
    
    
print('Dates that are not available')
for i in impossible:
    print(i)
    
 

Villa name              Dolphin
Month                  Apr
Date                    8
Number of days: 4
Dates that are available
8-Apr
9-Apr
Dates that are not available
10-Apr
11-Apr
